# SignSage — FDMSE-ISL 200-Word Recognition Training (Google Colab)

**Purpose.** Train a practical, reproducible **200-word isolated Indian Sign Language (ISL)** recognizer on Google Colab's free GPU using the FDMSE-ISL dataset, and export a bundle SignSage can load experimentally.

**What this is NOT**

- It does **not** claim continuous sentence translation. This is isolated word recognition only.
- It does **not** promote any model into SignSage's production pipeline. The exported bundle is an **experimental artifact** for evaluation first.
- It does **not** download the 30 GB dataset to your laptop. Everything runs in the Colab runtime.

**Before you run**

1. In Colab: **File → Upload notebook** and upload this `.ipynb` (or open it from GitHub/Drive with `Open in Colab`).
2. Select a **GPU** runtime: **Runtime → Change runtime type → GPU** (free tier usually assigns NVIDIA T4; any CUDA GPU works).
3. Provide the key as a **Colab secret** named `FDMSE_ISL_KEY` (key icon in the left sidebar → **Add new secret**) and tick **Notebook access**. **Never type the key into a cell.** If the secret is unavailable you can enter it at the secure hidden prompt instead.
4. **Runtime → Run all** (or run section by section).

**Vocabulary** — the target is 200 classes chosen automatically from the 2002 published classes, preferring classes present in all three published partitions (train 10 signers / validation 2 / test 8) with the strongest per-partition coverage. If fewer than 200 classes qualify, the notebook says so — it will not silently fake 200 classes.

**Time budget** — `MODE = "QUICK"` (default): ~48 landmark frames per video, ~8–12 epochs with early stopping. `MODE = "FULL"`: more epochs. Acquisition/preprocessing runs on CPU; training runs on GPU.

**Runtime lifetime** — Colab runtimes are temporary. Download the exported bundle from **Files** (folder icon) as soon as it finishes, or set `USE_DRIVE = True` in Cell 0 to sync checkpoints and the final bundle to Google Drive automatically.

**Legal** — FDMSE-ISL is research-use only. Do not redistribute the archive, the extracted videos, or this notebook with any key embedded. Cite the paper (see the last section).

In [ ]:
# =====================================================================
# Cell 0 — CONFIGURATION  (edit here, nothing else)
# =====================================================================
import os
from pathlib import Path

# ---- Reproducibility --------------------------------------------------
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- Mode ------------------------------------------------------------
MODE = "QUICK"          # "QUICK" | "FULL"
assert MODE in ("QUICK", "FULL")

# ---- Targets ---------------------------------------------------------
TARGET_WORDS = 200      # desired vocabulary size; eligibility report decides the real count

# ---- Vocabulary eligibility (all three partitions must have coverage) -
MIN_TRAIN_VIDEOS = 6
MIN_VAL_VIDEOS   = 2
MIN_TEST_VIDEOS  = 4
MIN_TRAIN_SIGNERS = 2   # a class used at eval time must be signed by >= N training signers

# ---- Preprocessing ---------------------------------------------------
N_LANDMARK_FRAMES = 48     # frames sampled per video BEFORE landmark extraction (speed)
SEQ_LEN           = 60     # frames fed to the model (sign_features_v3 resamples to 60)
FEAT_DIM          = 122    # sign_features_v3 output dim — do not change without retraining
CACHE_FEATURES    = True
KEEP_LANDMARKS    = False  # keep raw (T,543,3) .npy files after features are cached

# ---- Training --------------------------------------------------------
BATCH_SIZE      = 64
MAX_EPOCHS      = 12 if MODE == "QUICK" else 40
EARLY_PATIENCE  = 6
LEARNING_RATE   = 3e-4
WEIGHT_DECAY    = 0.05
LABEL_SMOOTHING = 0.1
DROPOUT         = 0.3
HIDDEN_SIZE     = 128
GRU_LAYERS      = 2
AMP             = True     # mixed precision on CUDA
NUM_WORKERS     = 2
RESUME          = True     # resume from ckpt_latest.pt if present (crash-safe)

# ---- Paths: /content = Colab runtime storage (LOCAL; lost on disconnect).
#             /content is local to the runtime and is lost on disconnect:
#             unsynced /content disappears — enable USE_DRIVE to persist outputs.
IS_COLAB  = Path("/content").exists()
WORK_ROOT = Path("/content") if IS_COLAB else Path("./colab_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# ---- Optional Google Drive persistence (default OFF) -------------------
# When True: checkpoints, metrics and the final bundle are synced to Drive
# after training/export, and a previous best checkpoint is copied back here
# on resume. Local /content keeps the heavy work (archive, subset, caches).
USE_DRIVE  = False
DRIVE_ROOT = Path("/content/drive/MyDrive/SignSage_FDMSE_training") if IS_COLAB else Path("./drive_fdmse")
# Optional consolidated feature-cache archive (Drive). Saved once per session end;
# restored at the start of the next session instead of re-running landmarks.
DRIVE_FEATURES_DIR   = (DRIVE_ROOT / "fdmse_features") if USE_DRIVE else None
DRIVE_FEATURE_SHARDS = (DRIVE_ROOT / "fdmse_features" / "shards") if USE_DRIVE else None
DRIVE_FEATURE_MANIFEST = (DRIVE_ROOT / "fdmse_features" / "manifest.csv") if USE_DRIVE else None

CKPT_DIR           = WORK_ROOT / "checkpoints"
BUNDLE_DIR         = WORK_ROOT / "fdmse_200word_bundle"
for d in (CKPT_DIR, BUNDLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ──Scratch paths are assigned in Cell 2 AFTER the remote size is known────
SCRATCH_ROOT       = None   # decided in Cell 2 from measured free space
ARCHIVE_PATH       = None   # SCRATCH_ROOT / "FDMSE-ISL.zip"
DATA_SUBSET        = None   # SCRATCH_ROOT / "fdmse_isl_subset"
LANDMARK_CACHE     = None   # SCRATCH_ROOT / "fdmse_landmarks"
FEATURE_CACHE      = None   # SCRATCH_ROOT / "fdmse_features"

CKPT_LATEST     = CKPT_DIR / "ckpt_latest.pt"
CKPT_BEST       = CKPT_DIR / "ckpt_best.pt"
HISTORY_PATH    = CKPT_DIR / "history.json"
MANIFEST_PATH   = CKPT_DIR / "manifest.csv"
VOCAB_JSON      = CKPT_DIR / "vocabulary.json"
LABEL_MAP_JSON  = CKPT_DIR / "label_mapping.json"
METRICS_JSON    = CKPT_DIR / "metrics.json"
VOCAB_REPORT    = CKPT_DIR / "vocabulary_report.txt"
FAILURES_JSON    = CKPT_DIR / "extraction_failures.json"

# ---- Dataset access (official portal format; key comes from a secret) -
# The official portal serves the archive over HTTPS (verified 2026-11:
# HTTP 403 on a bogus key proves TLS works). Plain HTTP is NOT used unless
# ALLOW_INSECURE_HTTP is explicitly set to True below.
DOWNLOAD_URL_TEMPLATE = "https://cs.rkmvu.ac.in/~isl/isl-dataset/file?key={key}"
ALLOW_INSECURE_HTTP   = False   # set True only if the portal ever drops HTTPS; a loud warning is printed
DOWNLOAD_URL_TEMPLATE_HTTP = "http://cs.rkmvu.ac.in/~isl/isl-dataset/file?key={key}"

# ---- Resume the archive download, if partially present ---------------
RESUME_DOWNLOAD = True    # set False to always restart the download from byte zero
MAX_RETRIES     = 6
CHUNK_MB        = 8
DOWNLOAD_TIMEOUT_S = 120
if ALLOW_INSECURE_HTTP:
    import warnings
    warnings.warn("ALLOW_INSECURE_HTTP=True — the access key will transit IN PLAIN TEXT over plain HTTP. "
                  "Plain HTTP is never used silently; this flag exists only as a manual escape hatch.", stacklevel=1)
    DOWNLOAD_URL_TEMPLATE = DOWNLOAD_URL_TEMPLATE_HTTP

# ---- Dataset view preference -----------------------------------------
PREFER_VIEW = "frontal"   # filter is APPLIED in Cell 6; anything else needs FORCE_VIEW
FORCE_VIEW  = None        # manual override, e.g. "left" / "right" / "side"; wins over PREFER_VIEW.

# ---- Published split (must be verified or the run stops) ---------------
EXPECTED_SIGNERS = {"train": 10, "validation": 2, "test": 8}   # FDMSE-ISL published 10/2/8 partition
EXPECTED_VIDEOS  = {"train": 20_016, "validation": 4_003, "test": 16_014}  # published counts (frontal 60fps working set)
ALLOW_SPLIT_MISMATCH = os.environ.get("FDMSE_ALLOW_SPLIT_MISMATCH", "").lower() in {"1", "true", "yes"}
print(f"MODE={MODE} | IS_COLAB={IS_COLAB} | WORK_ROOT={WORK_ROOT} | USE_DRIVE={USE_DRIVE}")
print(f"target_words={TARGET_WORDS} | frames={N_LANDMARK_FRAMES} | seq_len={SEQ_LEN} | feat_dim={FEAT_DIM}")
print(f"quick epochs={MAX_EPOCHS} | batch={BATCH_SIZE} | lr={LEARNING_RATE} | amp={AMP}")


In [ ]:
# =====================================================================
# Cell 1 — ENVIRONMENT, GPU, DISK, NETWORK CHECKS
# =====================================================================
import shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = Path("/content").exists() and (Path("/content").is_dir())
print("runtime    ", "Google Colab" if IN_COLAB else "local (not Colab) — tuned for Colab")

# Dependencies: install only what is missing (torch/numpy/pandas/matplotlib ship with Colab).
def ensure_import(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        __import__(import_name)
        return True
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        return True

ensure_import("requests")
ensure_import("tqdm")
ensure_import("pandas")
ensure_import("opencv-python-headless", "cv2")
ensure_import("scikit-learn", "sklearn")
ensure_import("mediapipe")
ensure_import("matplotlib")
try:
    import psutil
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "psutil"])

import cv2
import mediapipe as mp
import numpy as np
import psutil
import requests
import sklearn
import torch
import tqdm

print("python    ", sys.version.split()[0])
print("torch     ", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("mediapipe ", mp.__version__, "| cv2:", cv2.__version__, "| sklearn:", sklearn.__version__)
if torch.cuda.is_available():
    print("gpu       ", torch.cuda.get_device_name(0),
          f"(CUDA capability {torch.cuda.get_device_capability(0)})")
    vram = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"vram      {vram:.1f} GiB  → mixed precision: {'on (fp16 via torch.amp)' if vram >= 8 else 'on (fp16, watch memory)'}")
    print("note      Colab free GPUs are commonly NVIDIA T4 (16 GB); A100 on paid tiers works unchanged")
else:
    print("gpu       NONE — enable a GPU: Runtime > Change runtime type > GPU (T4 recommended)")
ram = psutil.virtual_memory()
print(f"ram       {ram.available / 2**30:.1f} GiB available (of {ram.total / 2**30:.1f} GiB)")

usage = shutil.disk_usage(WORK_ROOT)
free_gb = usage.free / 2**30
print(f"disk free  {free_gb:.1f} GB at {WORK_ROOT} (total {usage.total/2**30:.1f} GB)")
print(f"disk free  {shutil.disk_usage('/tmp').free / 2**30:.1f} GB at /tmp")
os.system("df -h /content /tmp 2>/dev/null || true")

# Network reachability (does NOT touch the dataset endpoint)
for host in ("https://cs.rkmvu.ac.in", "https://storage.googleapis.com"):
    try:
        requests.head(host, timeout=8, allow_redirects=True)
        print(f"net        {host}: reachable")
    except Exception as e:
        print(f"net        {host}: UNREACHABLE ({e.__class__.__name__}) — check Colab internet settings")

# ---- Optional Google Drive persistence (default OFF) -------------------
if USE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted — checkpoints/bundle will sync to {DRIVE_ROOT}")
    except Exception as e:
        raise SystemExit(
            f"Google Drive mount failed ({e.__class__.__name__}: {e}). "
            "Set USE_DRIVE=False in Cell 0 to use runtime storage only."
        )
else:
    print("Google Drive persistence OFF — artifacts stay in /content (lost on disconnect).")

def sync_to_drive(*paths):
    """Copy important OUTPUT files to Drive when USE_DRIVE=True; no-op otherwise."""
    if not USE_DRIVE:
        return
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in paths:
        p = Path(p)
        if p.is_file():
            shutil.copy2(p, DRIVE_ROOT / p.name)
            n += 1
    if n:
        print(f"synced {n} file(s) to {DRIVE_ROOT}")


## Section 2 — Secure dataset acquisition (remote download, no laptop copy)

The archive is downloaded directly into the Colab runtime and stays there (`/content` or the fastest local disk). The access key is read from a **Colab secret** (`FDMSE_ISL_KEY`, key icon in the sidebar), falling back to a hidden prompt. It is never printed, saved, or embedded in any output file.

The cell implements:

- `HEAD` check → remote size and `Accept-Ranges` support (falls back to a 1-byte `GET` probe if `HEAD` is unsupported),
- disk-space requirement check (`archive size × 1.15 + 4 GB margin`),
- **resumable** download (HTTP `Range` from a `.part` file) with retries and timeouts,
- progress output without ever printing the URL or key,
- integrity check of the archive (openable + sampled CRC validation).

If the archive cannot fit, this stops with a clear explanation instead of silently switching strategy.

In [ ]:
# =====================================================================
# Cell 2 — SECURE KEY HANDLING + PRE-FLIGHT CHECKS
# =====================================================================
import getpass, json

def get_access_key() -> str:
    # 1) Colab secrets (key icon in the left sidebar → "FDMSE_ISL_KEY", notebooks access enabled)
    try:
        from google.colab import userdata  # type: ignore
        key = (userdata.get("FDMSE_ISL_KEY") or "").strip()
        if key:
            return key
    except Exception:
        pass
    # 2) environment variable (handy for local runs)
    key = os.environ.get("FDMSE_ISL_KEY", "").strip()
    if key:
        return key
    # 3) hidden prompt (input not echoed, never stored)
    print("Colab secret 'FDMSE_ISL_KEY' not found. Paste the key at the hidden prompt (input not echoed).")
    return getpass.getpass("FDMSE-ISL access key: ").strip()

def head_info(url_template, key):
    """Return (status, content_length_bytes_or_None, accept_ranges_bool, content_type). Key stays local."""
    url = url_template.format(key=key)
    try:
        r = requests.head(url, timeout=30, allow_redirects=True)
        ok_head = r.status_code < 400
    except Exception:
        ok_head = False
        r = None
    if ok_head and r is not None:
        return r.status_code, _to_int(r.headers.get("Content-Length")), (r.headers.get("Accept-Ranges", "").lower() == "bytes"), r.headers.get("Content-Type")
    # Fallback: 1-byte ranged GET
    r = requests.get(url, headers={"Range": "bytes=0-0"}, stream=True, timeout=30)
    r.close()
    cr = r.headers.get("Content-Range", "")
    total = None
    if "/" in cr:
        try: total = int(cr.split("/")[-1])
        except ValueError: total = None
    return r.status_code, total, ("bytes" in r.headers.get("Accept-Ranges", "").lower() or total is not None), r.headers.get("Content-Type")

def _to_int(v):
    try: return int(v)
    except (TypeError, ValueError): return None

ACCESS_KEY = get_access_key()
if not ACCESS_KEY:
    raise SystemExit("No access key provided. Add the 'FDMSE_ISL_KEY' Colab secret (sidebar key icon, 'all notebooks' access) or paste it at the prompt, then re-run.")
print(f"Access key loaded: {len(ACCESS_KEY)} chars (contents not shown)")

status, remote_size, accept_ranges, content_type = head_info(DOWNLOAD_URL_TEMPLATE, ACCESS_KEY)
scheme = DOWNLOAD_URL_TEMPLATE.split("://", 1)[0].lower()
print(f"scheme           : {scheme}")
print(f"portal status    : {status}")
print(f"content-type     : {content_type}")
print(f"remote size      : {remote_size / 2**30:.2f} GB" if remote_size else "remote size      : UNKNOWN (portal did not report Content-Length)")
print(f"range requests   : {'supported' if accept_ranges else 'not reported'}")
if scheme != "https":
    print("WARNING: NOT downloading over HTTPS. The access key WOULD transit in plain text.")
    if not ALLOW_INSECURE_HTTP:
        raise SystemExit("Refusing plain-HTTP download. The portal supports HTTPS (verified). Set DOWNLOAD_URL_TEMPLATE to https. "
                         "To override deliberately, set ALLOW_INSECURE_HTTP=True in Cell 0 — you will get this warning in the open, never silently.")
print("transport        : HTTPS (key in URL query string is the portal's design; TLS protects it in transit — do not share the URL)")

# ---- Select SCRATCH storage from MEASURED free space -------------------
print("\nmeasuring candidate scratch locations (df -h style):")
candidates = ["/tmp", "/content", "./scratch"]  # /tmp is ephemeral and is NOT synced; /content is the runtime disk
measurements = []
for c in candidates:
    p = Path(c)
    try:
        usage = shutil.disk_usage(p if p.exists() else Path("."))
        print(f"  {c:20s} total={usage.total / 2**30:6.1f} GB  free={usage.free / 2**30:6.1f} GB")
        measurements.append((p, usage.free))
    except OSError:
        print(f"  {c:20s} not measurable (skipped)")
print("\n(df -h on the mount chosen below, for reference:)")
os.system("df -h /tmp 2>/dev/null || true")

# Disk plan: archive itself + extracted subset (~4-6 GB) + feature cache (<1 GB)
# + slack. The download is forbidden to start if this cannot fit.
_SUBSET_ESTIMATE_GIB = 6.0
_CACHE_ESTIMATE_GIB = 1.0
_SLACK_GIB = 4.0
required_gib = (remote_size / 2**30 if remote_size else 33.0) + _SUBSET_ESTIMATE_GIB + _CACHE_ESTIMATE_GIB + _SLACK_GIB
print(f"\nspace required   : ~{required_gib:.1f} GB  (archive {remote_size / 2**30 if remote_size else 33.0:.1f} GB + subset {_SUBSET_ESTIMATE_GIB:.0f} GB + cache {_CACHE_ESTIMATE_GIB:.0f} GB + slack {_SLACK_GIB:.0f} GB)")
required_gb = required_gib
working_free = shutil.disk_usage(WORK_ROOT).free / 2**30
print(f"working dir free : {working_free:.1f} GB  (checkpoints + bundle live here)")

best_p, best_free = (Path("./scratch"), 0)
for p, free in measurements:
    if free > best_free:
        best_p, best_free = p, free

SCRATCH_ROOT = best_p
scratch_free_gb = best_free / 2**30
if scratch_free_gb < required_gb:
    if working_free >= required_gb:
        SCRATCH_ROOT = WORK_ROOT
        print(f"WARNING: no scratch location fits {required_gb:.1f} GB; falling back to /content "
              f"({working_free:.1f} GB free). The archive will consume most of the runtime disk.")
    else:
        raise SystemExit(
            f"NOT ENOUGH DISK: best scratch {SCRATCH_ROOT} has {scratch_free_gb:.1f} GB and /content has {working_free:.1f} GB; "
            f"need ~{required_gb:.1f} GB. Free space (delete other datasets) and re-run. The download is NOT started."
        )
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH       = SCRATCH_ROOT / "FDMSE-ISL.zip"
DATA_SUBSET        = SCRATCH_ROOT / "fdmse_isl_subset"
LANDMARK_CACHE     = SCRATCH_ROOT / "fdmse_landmarks"
FEATURE_CACHE      = SCRATCH_ROOT / "fdmse_features"
for d in (DATA_SUBSET, LANDMARK_CACHE, FEATURE_CACHE):
    d.mkdir(parents=True, exist_ok=True)
print(f"scratch selected : {SCRATCH_ROOT}  ({shutil.disk_usage(SCRATCH_ROOT).free / 2**30:.1f} GB free)")
print(f"archive          : {ARCHIVE_PATH}")
print(f"subset + caches  : {DATA_SUBSET}, {FEATURE_CACHE}")
print("durable artifacts (checkpoints/bundle/metrics):", WORK_ROOT)


In [ ]:
# =====================================================================
# Cell 3 — RESUMABLE DOWNLOAD (progress without exposing the key)
# =====================================================================
def _expected_content_range_ok(cr: str, existing: int, total) -> bool:
    """Validate 'bytes <start>-<end>/<total>' matches resume expectations."""
    try:
        unit, spec = cr.split(" ", 1)
        if unit.lower() != "bytes":
            return False
        start, rest = spec.split("-", 1)
        end, total_s = rest.split("/", 1)
        if int(start) != existing:
            return False
        if total is not None and int(total_s) != total:
            return False
        return int(end) >= int(start)
    except Exception:
        return False


def download_with_resume(url_template, key, dest: Path, expected_size, resume: bool = True):
    url = url_template.format(key=key)
    part = dest.with_suffix(dest.suffix + ".part")
    expected_total = expected_size

    existing = part.stat().st_size if (resume and part.exists()) else 0
    if existing:
        if expected_total and existing >= expected_total:
            # Do not trust size alone: re-check the remote size, then accept.
            status2, size2, _acc, _ct = head_info(url_template, key)
            if size2 == existing:
                print(f"partial file is already complete ({existing:,} bytes) and matches remote size — verifying and finalizing")
                part.replace(dest)
                return dest
            raise SystemExit(
                f"partial file ({existing:,} bytes) is >= expected size ({expected_total:,} bytes) but the portal "
                f"reports remote size {size2:,} bytes — refusing to finalize a mismatched file. Delete {part} to restart."
            )
        print(f"partial download found: {existing / 2**20:.1f} MB — asking the portal to resume")

    for attempt in range(1, MAX_RETRIES + 1):
        headers = {}
        if existing:
            headers["Range"] = f"bytes={existing}-"
        try:
            with requests.get(url, headers=headers, stream=True, timeout=(30, DOWNLOAD_TIMEOUT_S)) as r:
                if r.status_code == 416 and existing and expected_total and existing >= expected_total:
                    # Strict recovery: only accept the partial after a FRESH HEAD confirms it == remote size.
                    status2, size2, _acc, _ct = head_info(url_template, key)
                    if size2 and int(size2) == existing:
                        print(f"server reported 416 and HEAD confirms remote size == partial ({existing:,} B) — finalizing verified partial")
                        part.replace(dest)
                        return dest
                    raise SystemExit(
                        f"HTTP 416 and partial is {existing:,} B, but HEAD reports remote size {size2} B — refusing to finalize a mismatched file. "
                        f"Inspect/backup {part}, verify the archive via Cell 4, or delete the .part file and re-download."
                    )
                if r.status_code not in (200, 206):
                    print(f"attempt {attempt}: portal returned HTTP {r.status_code}")
                    continue

                # ---- resume-vs-restart decision, made BEFORE writing anything ----
                if existing:
                    if r.status_code == 200:
                        print(f"server did NOT honor the Range header (HTTP 200) — restarting from zero")
                        part.unlink(missing_ok=True)
                        existing = 0
                    else:  # 206 — Content-Range must be present and match our offset exactly
                        cr = r.headers.get("Content-Range", "")
                        if not _expected_content_range_ok(cr, existing, expected_total):
                            raise SystemExit(
                                f"206 response has invalid/mismatched Content-Range {cr!r} for a resume from byte "
                                f"{existing}. Refusing to append (would corrupt the archive) and keeping the partial "
                                f"file at {part}. Inspect the portal or re-run with the partial deleted to restart cleanly."
                            )

                if existing:
                    mode = "ab"
                else:
                    mode = "wb"
                downloaded = existing
                next_report = downloaded + 256 * 2**20
                with open(part, mode) as fh:
                    for chunk in r.iter_content(chunk_size=CHUNK_MB * 2**20):
                        if not chunk:
                            continue
                        fh.write(chunk)
                        downloaded += len(chunk)
                        if downloaded >= next_report:
                            next_report = downloaded + 512 * 2**20
                            if expected_total:
                                pct = 100 * downloaded / expected_total
                                print(f"  downloaded {downloaded / 2**30:6.2f} GB / {expected_total / 2**30:.2f} GB ({pct:5.1f}%)")
                            else:
                                print(f"  downloaded {downloaded / 2**30:6.2f} GB")
                break  # success
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError,
                requests.exceptions.Timeout, requests.exceptions.RequestException) as e:
            print(f"attempt {attempt}: connection dropped ({e.__class__.__name__}); resuming…")
            existing = part.stat().st_size if part.exists() else 0
            continue
    else:
        raise SystemExit("Download failed after retries. Check Internet, the access key, and the portal, then re-run — the .part file is preserved for resume.")

    part.replace(dest)
    return dest

PARTIAL = ARCHIVE_PATH.with_suffix(".zip.part")
if ARCHIVE_PATH.exists():
    print(f"Archive already present at {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 2**30:.2f} GB) — skipping download.")
else:
    download_with_resume(DOWNLOAD_URL_TEMPLATE, ACCESS_KEY, ARCHIVE_PATH, remote_size, RESUME_DOWNLOAD)
    size_gb = ARCHIVE_PATH.stat().st_size / 2**30
    print(f"Download complete: {size_gb:.2f} GB")
    if remote_size and ARCHIVE_PATH.stat().st_size != remote_size:
        raise SystemExit(f"Size mismatch: portal said {remote_size / 2**30:.2f} GB, file is {size_gb:.2f} GB. Delete {ARCHIVE_PATH} and re-run.")


In [ ]:
# =====================================================================
# Cell 4 — ARCHIVE INTEGRITY CHECK (open + sampled CRC validation)
# =====================================================================
import json, random, zipfile

if not ARCHIVE_PATH.exists():
    raise SystemExit(f"Archive missing: {ARCHIVE_PATH}")

with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    infos = zf.infolist()
    names = [i.filename for i in infos]
    video_exts = {".mp4", ".mov", ".mkv", ".avi", ".webm"}
    videos = [n for n in names if Path(n).suffix.lower() in video_exts]
    others = [n for n in names if Path(n).suffix.lower() not in video_exts]
    zero_byte = [i for i in infos if i.file_size == 0]
    print(f"entries      : {len(infos)}  (videos: {len(videos)}, other files: {len(others)})")
    print(f"zero-byte    : {len(zero_byte)}")
    # Integrity: sampled CRC check (fast). Full archive decompression check is available but
    # takes many minutes at this size; set VERIFY_FULL_CRC=True only if you suspect corruption.
    VERIFY_FULL_CRC = False
    if VERIFY_FULL_CRC:
        bad = zf.testzip()
        print("zip integrity:", "OK" if bad is None else f"CORRUPT at {bad}")
        if bad is not None:
            raise SystemExit("Archive failed CRC validation — delete it and re-download.")
    else:
        import random as _rnd
        sample = _rnd.Random(SEED).sample(infos, min(25, len(infos)))
        bad_entries = []
        for info in sample:
            try:
                with zf.open(info) as fh:
                    _ = fh.read()
            except Exception:
                bad_entries.append(info.filename)
        print(f"crc spotcheck: {len(sample) - len(bad_entries)}/{len(sample)} sampled entries OK")
        if bad_entries:
            raise SystemExit(f"Corrupt entries detected: {bad_entries[:5]}. Delete the archive and re-download.")
        print("zip integrity: sampled CRC check OK (full decompression check not run; set VERIFY_FULL_CRC=True to force it)")
    ext_hist = {}
    for n in names:
        ext = Path(n).suffix.lower() or "(none)"
        ext_hist[ext] = ext_hist.get(ext, 0) + 1
    print("extensions   :", dict(sorted(ext_hist.items(), key=lambda kv: -kv[1])))
    meta_files = [n for n in names if Path(n).suffix.lower() in {".csv", ".json", ".txt", ".md", ".yaml", ".yml", ".xml"}]
    print(f"metadata     : {len(meta_files)} file(s)")
    for n in meta_files[:40]:
        print("   ", n)
    Path(CKPT_DIR / "archive_index.json").write_text(
        json.dumps({"entries": len(infos), "videos": len(videos), "metadata_files": meta_files}, ensure_ascii=False, indent=2)
    )
print("Archive OK. Listing saved to checkpoints/archive_index.json")


## Section 3 — Dataset inspection (no assumptions)

The notebook does **not** assume the archive layout. It:

1. Scans every video entry (ZIP central directory — nothing extracted yet),
2. scans metadata files inside the archive first (CSV/JSON/YAML manifests, if any — they usually carry `word`, `signer`, `split` columns),
3. falls back to path-based discovery (scoring each path component for the role: `split`, `signer`, `word`, `view`),
4. prints a full report: detected layout, per-class counts, signer coverage, corrupt/zero-byte files.

**Split is never reassigned randomly.** FDMSE-ISL is distributed with signer-independent partitions (10 train / 2 validation / 8 test signers). The notebook *discovers and respects* them. If the discovered partitions share a signer, the notebook **stops** — it will not silently produce an invalid evaluation.

Config knobs at the top of the next cell let you pin the layout manually if auto-detection is ambiguous.

In [ ]:
# =====================================================================
# Cell 5 — STRUCTURE DISCOVERY (zip listing → parsed table)
# Manual overrides (None = auto-detect):
# =====================================================================
FORCE_SPLIT_INDEX  = None
FORCE_SIGNER_INDEX = None
FORCE_WORD_INDEX   = None
SPLIT_MARKERS = {
    "train":      {"train", "training", "tr"},
    "validation": {"val", "valid", "validation", "dev"},
    "test":       {"test", "testing", "te"},
}
SPLIT_FIT = 0.50   # fraction of entries a component must match to be accepted as the split marker
VIEW_TOKENS = ("front", "frontal", "left", "right", "view", "cam", "rgb", "depth", "kinect")
VIDEO_EXTS  = {".mp4", ".mov", ".mkv", ".avi", ".webm"}

import re
import sys
import pandas as pd

def norm_parts(name: str):
    return [p for p in name.replace("\\", "/").split("/") if p not in ("", ".")]

with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    entries = zf.infolist()
video_entries = [i for i in entries if Path(i.filename).suffix.lower() in VIDEO_EXTS]
nonvideo_entries = [i for i in entries if Path(i.filename).suffix.lower() not in VIDEO_EXTS]
print(f"zip entries: {len(entries)} | video files: {len(video_entries)}")

# ---------- 1) try metadata-first ------------------------------------
meta_cands = [i.filename for i in nonvideo_entries
              if Path(i.filename).suffix.lower() in {".csv", ".json", ".txt", ".yaml", ".yml"}]
print(f"metadata candidate files: {len(meta_cands)}")
for n in meta_cands[:25]:
    print("   ", n)

def from_metadata():
    """Best-effort manifest parse. Returns DataFrame or None."""
    import io, json as _json
    for name in meta_cands:
        if Path(name).suffix.lower() != ".csv":
            continue
        try:
            with zipfile.ZipFile(ARCHIVE_PATH) as zf:
                df = pd.read_csv(io.BytesIO(zf.read(name)))
        except Exception:
            continue
        cols = {c.lower().strip(): c for c in df.columns}
        def pick(*kws):
            for kw in kws:
                for low, orig in cols.items():
                    if kw in low:
                        return orig
            return None
        f = pick("file", "video", "path", "filename")
        w = pick("word", "gloss", "label", "sign")
        s = pick("signer", "subject", "person", "subj")
        p = pick("split", "partition", "set", "phase")
        v = pick("view", "camera", "cam", "angle", "viewpoint")
        if not (f and w and s and p):
            continue
        out = pd.DataFrame({
            "path": df[f].astype(str), "word": df[w].astype(str).str.strip().str.lower(),
            "signer": df[s].astype(str).str.extract(r"(\d+)", expand=False).fillna(df[s].astype(str)),
            "split": df[p].astype(str).str.strip().str.lower().map(
                lambda v: "train" if v in ("train", "training", "tr") else
                          ("validation" if v in ("val", "valid", "validation", "dev") else
                           ("test" if v in ("test", "testing", "te") else v))),
        })
        if v:
            out["view"] = df[v].astype(str).str.strip().str.lower()
        else:
            out["view"] = "unknown"
        cov = out["path"].isin([i.filename for i in video_entries]).mean()
        print(f"metadata manifest {name}: {len(out)} rows, path coverage {cov:.1%}")
        if cov >= 0.8:
            return out
    return None

meta_df = from_metadata()

def path_based():
    rows = []
    for i in video_entries:
        parts = norm_parts(i.filename)
        rows.append((i.filename, parts))
    max_depth = max((len(p) for _, p in rows), default=0)

    def split_at(idx):
        ok = 0
        for _, p in rows:
            if idx < len(p) and p[idx].strip().lower() in set().union(*SPLIT_MARKERS.values()):
                ok += 1
        return ok / len(rows) if rows else 0.0

    split_scores = [split_at(i) for i in range(max_depth)]
    split_idx = int(np.argmax(split_scores)) if split_scores else None
    if split_scores and split_scores[split_idx] < SPLIT_FIT:
        split_idx = None
    print("split candidate scores per depth:", [f"{s:.2f}" for s in split_scores])

    def signer_at(idx):
        pat = re.compile(r"^(?:s|signer|subj|subject|user|person)?[-_.]?\d{1,4}$", re.I)
        ok, vals = 0, set()
        for _, p in rows:
            if idx < len(p) and pat.match(p[idx].strip()):
                ok += 1
                vals.add(p[idx].strip().lower())
        return ok / len(rows), len(vals)

    signer_scores, signer_ns = [], []
    for i in range(max_depth):
        s, n = signer_at(i)
        signer_scores.append(s); signer_ns.append(n)
    # pick the depth with good coverage and a plausible signer count
    signer_idx = None
    best = -1.0
    for i in range(max_depth):
        s, n = signer_scores[i], signer_ns[i]
        if s >= 0.5 and 2 <= n <= 40:
            score = s * (1.0 / (1 + abs(20 - n)))
            if score > best:
                best, signer_idx = score, i
    print("signer candidate scores per depth:", [f"{s:.2f}(n={n})" for s, n in zip(signer_scores, signer_ns)])

    if FORCE_SPLIT_INDEX is not None: split_idx = FORCE_SPLIT_INDEX
    if FORCE_SIGNER_INDEX is not None: signer_idx = FORCE_SIGNER_INDEX

    view_frac = []
    word_nunique = []
    for idx in range(max_depth):
        vals = [p[idx] for _, p in rows if idx < len(p)]
        view_frac.append(float(np.mean([any(t in v.lower() for t in VIEW_TOKENS) for v in vals])) if vals else 1.0)
        word_nunique.append(len({p[idx].lower() for _, p in rows if idx < len(p)}))
    role = {split_idx, signer_idx}
    word_idx = None
    best = -1
    for idx in range(max_depth):
        if idx in role or idx == max_depth - 1 or view_frac[idx] > 0.5:
            continue
        nu = word_nunique[idx]
        if nu >= 2 and nu > best:
            best, word_idx = nu, idx
    print("view fraction per depth:   ", [f"{v:.2f}" for v in view_frac])
    print("word nunique per depth:    ", word_nunique, "-> word_idx =", word_idx)
    if FORCE_WORD_INDEX is not None:
        word_idx = FORCE_WORD_INDEX

    out_rows = []
    for name, p in rows:
        low = [x.lower() for x in p]
        view = next((x for x in low if any(t in x for t in VIEW_TOKENS)), "unknown")
        split = next((v for v, toks in SPLIT_MARKERS.items() if any(x in toks for x in low)), None)
        signer = re.sub(r"\D", "", p[signer_idx]) if signer_idx is not None and signer_idx < len(p) else None
        word = p[word_idx] if word_idx is not None and word_idx < len(p) else None
        out_rows.append((name, word, signer, split, view))
    df = pd.DataFrame(out_rows, columns=["path", "word", "signer", "split", "view"])
    return df

if meta_df is not None:
    df = meta_df
    PARSE_METHOD = "metadata-manifest"
else:
    df = path_based()
    PARSE_METHOD = "path-based"

if FORCE_WORD_INDEX is not None:
    df["word"] = [norm_parts(p)[FORCE_WORD_INDEX] if FORCE_WORD_INDEX < len(norm_parts(p)) else None for p in df["path"]]

df["word"] = df["word"].astype(str).str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
df = df.dropna(subset=["word", "signer", "split"])
df = df[df["split"].isin(["train", "validation", "test"])].reset_index(drop=True)
df.to_csv(CKPT_DIR / "parsed_index_full.csv", index=False)
print(f"\nparse method: {PARSE_METHOD}")
print(f"parsed rows  : {len(df)}")
print("split counts :", df["split"].value_counts().to_dict())
print("unique words :", df["word"].nunique(), "| unique signers:", df["signer"].nunique())
print("view values  :", df["view"].value_counts().head(5).to_dict())
for split in ("train", "validation", "test"):
    s = df.loc[df.split == split, "signer"]
    print(f"  {split:10s}: {len(s)} videos, {s.nunique()} signers, words={df.loc[df.split == split, 'word'].nunique()}")
print("(signer-overlap and published-split verification run in Cell 6, after view filtering)")
print("\nfirst parsed rows:")
print(df.head(8).to_string())


In [ ]:
# =====================================================================
# Cell 6 — CLASS DISTRIBUTION + VIEW SELECTION + CORRUPT FILES
# =====================================================================
print("videos per word (top/least sampled example classes):")
counts = df.groupby("word").size().sort_values()
print(counts.head(8).to_dict())
print("...", counts.iloc[len(counts)//2], "median ...", counts.iloc[-8:].to_dict())

# Camera selection: PREFER_VIEW is applied; unknown layout STOPS for a manual override.
views = df["view"].str.lower().fillna("unknown")
print("views present:", views.value_counts().to_dict())
if FORCE_VIEW:
    keep = views.str.contains(FORCE_VIEW.lower(), regex=False)
    matches = sorted(views[keep].unique())
    if not keep.any():
        raise SystemExit(
            f"FORCE_VIEW={FORCE_VIEW!r} matched nothing. Available view tokens: {sorted(views.unique())}. "
            "Set FORCE_VIEW to one of those tokens to proceed."
        )
    dropped = int((~keep).sum())
    print(f"manual view override FORCE_VIEW={FORCE_VIEW!r}: kept {int(keep.sum())} clips ({matches}), dropped {dropped}")
    df = df[keep].reset_index(drop=True)
else:
    if PREFER_VIEW:
        front = views.str.contains(PREFER_VIEW.lower(), regex=False) | (views == "frontal")
        if not front.any():
            available = sorted(views.unique())
            if available == ["unknown"]:
                # Public FDMSE-ISL release is the frontal working set (portal notes);
                # no camera tokens exist to disambiguate. Proceed on that basis — never
                # silently: this is printed loudly, and FORCE_VIEW remains available.
                print("NO camera-view tokens anywhere in this archive.")
                print("Assumption: public FDMSE-ISL release currently distributes the frontal RGB view only.")
                if sys.stdin is None or not sys.stdin.isatty():
                    raise SystemExit(
                        "Camera view cannot be established: no 'frontal' view token was found anywhere in the archive "
                        "AND the session is non-interactive. Set FORCE_VIEW=<token> in Cell 0 to the true view, "
                        "or PREFER_VIEW=None if that is genuinely the intended dataset, and re-run."
                    )
                confirm = input("Type 'yes' to proceed with the frontal-RGB assumption; anything else aborts: ").strip().lower()
                if confirm not in ("yes", "y"):
                    raise SystemExit(
                        "Aborted — camera view unresolved. Inspect the archive layout (Cell 5 prints it), "
                        "set FORCE_VIEW=<token> or PREFER_VIEW=None, and re-run."
                    )
                print("User confirmation received: proceeding with the frontal-RGB assumption.")
            else:
                raise SystemExit(
                    f"STOPPING: could not establish the {PREFER_VIEW!r} camera view. View tokens detected: {available}. "
                    f"Inspect the archive (Cell 5 prints it), then set FORCE_VIEW=<token> or PREFER_VIEW=None. "
                    "Refusing to mix camera views silently."
                )
        else:
            dropped = int((~front).sum())
            print(f"camera filter PREFER_VIEW={PREFER_VIEW!r}: kept {int(front.sum())} frontal clips, dropped {dropped} non-frontal")
            df = df[front].reset_index(drop=True)
    else:
        print("view filtering disabled (PREFER_VIEW=None); using all clips — report this in any write-up")

zero_bytes = [i.filename for i in entries if i.file_size == 0 and i.filename in set(df["path"])]
print(f"zero-byte videos in parsed set: {len(zero_bytes)}")
if zero_bytes:
    print("dropping zero-byte files:", zero_bytes[:10])
    df = df[~df["path"].isin(zero_bytes)].reset_index(drop=True)

# ---- Structural check first: signer disjointness across partitions --------
sig_sets = {s: set(df.loc[df.split == s, "signer"]) for s in ("train", "validation", "test")}
_leaks = []
if sig_sets["train"] & sig_sets["validation"]:
    _leaks.append(f"train ∩ validation = {sorted(sig_sets['train'] & sig_sets['validation'])}")
if sig_sets["train"] & sig_sets["test"]:
    _leaks.append(f"train ∩ test = {sorted(sig_sets['train'] & sig_sets['test'])}")
if sig_sets["validation"] & sig_sets["test"]:
    _leaks.append(f"validation ∩ test = {sorted(sig_sets['validation'] & sig_sets['test'])}")
if _leaks:
    raise SystemExit(
        "SIGNER LEAK ACROSS PARTITION BOUNDARIES — the signer-independent split is violated. "
        + "; ".join(_leaks)
        + ". This invalidates signer-independent evaluation. Fix the parse (FORCE_* overrides in Cell 5) and re-run. "
        "Do not bypass this check."
    )
print("signer overlap check: PASS (no shared signer across partitions)")

# ---- Published-split verification (signer counts must match exactly) ----
# Checked AFTER view filtering and zero-byte drop: the published 10/2/8 partition
# is a property of the frontal 60fps working set.
print("\npublished split verification (frontal working set — FDMSE-ISL paper):")
for split in ("train", "validation", "test"):
    sig = int(df.loc[df.split == split, "signer"].nunique()) if (df.split == split).any() else 0
    vid = int((df.split == split).sum())
    print(f"  {split:10s}: signers {sig:2d} (expected {EXPECTED_SIGNERS[split]}) | videos {vid:6d} (expected {EXPECTED_VIDEOS[split]})")
sig_ok = all(int(df.loc[df.split == s, "signer"].nunique()) == EXPECTED_SIGNERS[s] for s in ("train", "validation", "test"))
total_detected = int(len(df))
total_expected = sum(EXPECTED_VIDEOS.values())
vid_ratio = total_detected / total_expected
vid_ok = 0.8 <= vid_ratio <= 1.02
print(f"signer partition check: {'PASS' if sig_ok else 'FAIL'} (exactly 10/2/8 required)")
print(f"video count check    : {'PASS' if vid_ok else 'DEVIATION'} ({total_detected} detected vs {total_expected} published, ratio {vid_ratio:.2f})")
if not sig_ok or not vid_ok:
    if ALLOW_SPLIT_MISMATCH:
        print("WARNING: continuing despite split mismatch because FDMSE_ALLOW_SPLIT_MISMATCH is set — never set this in real runs.")
    else:
        raise SystemExit(
            "Split verification FAILED — detected partitioning does not match the published FDMSE-ISL "
            "signer-independent split (10 train / 2 validation / 8 test signers). Likely causes: partial/corrupt "
            "archive, or mis-detected split/signer levels. Inspect checkpoints/parsed_index_full.csv, fix the FORCE_* "
            "overrides in Cell 5, and re-run. Set FDMSE_ALLOW_SPLIT_MISMATCH=1 only to debug deliberately."
        )

assert (sig_ok and vid_ok) or ALLOW_SPLIT_MISMATCH

df.to_csv(CKPT_DIR / "parsed_index.csv", index=False)

# Duration / fps sample (30 random videos) — used only for a sanity check, not training
import cv2, random
sample = random.Random(SEED).sample(list(df["path"]), min(30, len(df)))
with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    durations, fps_vals = [], []
    for member in sample:
        tmp = WORK_ROOT / "_probe.mp4"
        with zf.open(member) as src, open(tmp, "wb") as dst:
            shutil.copyfileobj(src, dst)
        cap = cv2.VideoCapture(str(tmp))
        f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS) or 0
        if fps and f: durations.append(f / fps); fps_vals.append(fps)
        cap.release(); tmp.unlink()
print(f"sample durations (s): min={min(durations):.1f} median={np.median(durations):.1f} max={max(durations):.1f} | median fps={np.median(fps_vals):.0f}")

print(f"\nclass distribution: n={df['word'].nunique()} classes | min/median/max videos per class:",
      counts.min(), int(counts.median()), counts.max())
print(f"total videos (this view): {len(df)} | partitions: {df.split.value_counts().to_dict()}")


## Section 4 — Vocabulary selection (200 words)

Selection is automatic, deterministic, and based on:

- per-partition video counts (train / validation / test),
- per-partition signer diversity,
- balance (a class must be present in all three partitions),
- a single ranking score = `min(videos in the three partitions)`, then total videos, then signer spread.

The full eligibility report is printed and saved to `vocabulary_report.txt`. If fewer than 200 classes pass the thresholds, **that smaller number is trained and reported** — never silently 200.

Rows with no usable coverage are dropped from consideration; the mapping is saved for exact reproducibility.

In [ ]:
# =====================================================================
# Cell 7 — VOCABULARY SELECTION (deterministic, eligibility-ranked)
# =====================================================================
per = df.groupby(["word", "split"]).size().unstack(fill_value=0)
for col in ("train", "validation", "test"):
    if col not in per.columns:
        per[col] = 0
signers = df.groupby(["word", "split"])["signer"].nunique().unstack(fill_value=0)
for col in ("train", "validation", "test"):
    if col not in signers.columns:
        signers[col] = 0

eligible = []
for word, row in per.iterrows():
    tr, va, te = int(row["train"]), int(row["validation"]), int(row["test"])
    if tr < MIN_TRAIN_VIDEOS or va < MIN_VAL_VIDEOS or te < MIN_TEST_VIDEOS:
        continue
    if int(signers.loc[word, "train"]) < MIN_TRAIN_SIGNERS:
        continue
    score = min(tr, va, te)
    eligible.append({
        "word": word, "train": tr, "validation": va, "test": te,
        "total": tr + va + te,
        "signers": int(signers.loc[word].sum()),
        "score": score,
    })
elig_df = pd.DataFrame(eligible).sort_values(
    ["score", "total", "signers"], ascending=False, kind="mergesort"
).reset_index(drop=True)
print(f"classes with sufficient coverage in all partitions: {len(elig_df)}")

if len(elig_df) == 0:
    raise SystemExit(
        f"No class meets MIN_TRAIN={MIN_TRAIN_VIDEOS}, MIN_VAL={MIN_VAL_VIDEOS}, MIN_TEST={MIN_TEST_VIDEOS}. "
        "Inspect the distribution printout above — the archive layout may not match the parser. "
        "Set FORCE_* overrides in Cell 5 or lower the thresholds in Cell 0."
    )

selected_n = min(TARGET_WORDS, len(elig_df))
if selected_n < TARGET_WORDS:
    print(f"WARNING: only {selected_n} classes meet the thresholds — training on {selected_n}, not {TARGET_WORDS}.")
selected = elig_df.head(selected_n).reset_index(drop=True)
labels = list(selected["word"])
label2id = {w: i for i, w in enumerate(labels)}

json_map = {
    "label_list": labels,
    "label_to_id": label2id,
    "selection": {
        "target": TARGET_WORDS,
        "selected": selected_n,
        "criteria": {
            "min_train_videos": MIN_TRAIN_VIDEOS,
            "min_val_videos": MIN_VAL_VIDEOS,
            "min_test_videos": MIN_TEST_VIDEOS,
            "min_train_signers": MIN_TRAIN_SIGNERS,
        },
        "partition_videos": {row.word: {"train": int(row.train), "validation": int(row.validation), "test": int(row.test)} for row in selected.itertuples()},
    },
}
VOCAB_JSON.write_text(json.dumps(json_map, ensure_ascii=False, indent=2), encoding="utf-8")
LABEL_MAP_JSON.write_text(json.dumps({"labels": labels}, ensure_ascii=False, indent=2), encoding="utf-8")
report_lines = [
    "FDMSE-ISL vocabulary selection report",
    f"method=path/metadata-parsed | target={TARGET_WORDS} | selected={selected_n} | eligible={len(elig_df)}",
    f"thresholds: train>={MIN_TRAIN_VIDEOS}, validation>={MIN_VAL_VIDEOS}, test>={MIN_TEST_VIDEOS}, train_signers>={MIN_TRAIN_SIGNERS}",
    "",
    "word,train,validation,test,total,signers_selected_by_partition",
]
for row in selected.itertuples():
    report_lines.append(f"{row.word},{row.train},{row.validation},{row.test},{row.total},{row.signers}")
VOCAB_REPORT.write_text("\n".join(report_lines), encoding="utf-8")

for split in ("train", "validation", "test"):
    mask = df.split == split
    sub_words = df.loc[mask, "word"]
    sel_mask = sub_words.isin(label2id)
    print(f"{split:10s}: {int(sel_mask.sum())} videos across {sub_words[sel_mask].nunique()} classes")

df = df[df["word"].isin(label2id)].reset_index(drop=True)
print(f"\nselected {selected_n} classes, {len(df)} videos total")
print("least-covered selected classes (min partition count):")
print(selected.tail(5)[["word", "train", "validation", "test"]].to_string())
print("counts per partition for selected set:", df.split.value_counts().to_dict())
meta_out = dict(json.loads(VOCAB_JSON.read_text(encoding="utf-8")))
meta_out["n_videos_selected"] = len(df)
meta_out["parse_method"] = PARSE_METHOD
VOCAB_JSON.write_text(json.dumps(meta_out, ensure_ascii=False, indent=2), encoding="utf-8")


## Section 5 — Extract only the selected subset

Only the videos for the selected classes are extracted (typically ~10% of the archive). Extracted paths preserve the original relative layout. Already-extracted files are skipped on re-run, so restarting after a crash does not re-extract.

**The archive is kept** so that restarting the notebook never re-downloads the 30 GB archive. If disk is tight, set `ARCHIVE_PATH.unlink()` after successful extraction yourself — the notebook does not silently delete it.

In [ ]:
# =====================================================================
# Cell 8 — SELECTIVE EXTRACTION (zip → DATA_SUBSET)
# =====================================================================
selected_paths = set(df["path"])
print(f"extracting {len(selected_paths)} videos for {len(label2id)} classes…")

# Disk check BEFORE extraction: subset size + slack must fit the scratch filesystem.
extracted_bytes = 0
with zipfile.ZipFile(ARCHIVE_PATH) as _zf:
    _infos = {i.filename: i.file_size for i in _zf.infolist()}
extracted_bytes = sum(_infos.get(p, 0) for p in selected_paths)
_scratch_free = shutil.disk_usage(DATA_SUBSET).free
_required = extracted_bytes + 2 * 2**30   # subset + 2 GB slack for staging
print(f"extraction estimate: {extracted_bytes / 2**30:.2f} GB into {DATA_SUBSET} ({_scratch_free / 2**30:.1f} GB free); requiring ~{_required / 2**30:.2f} GB")
if _scratch_free < _required:
    raise SystemExit(
        f"NOT ENOUGH DISK to extract the subset: need ~{_required / 2**30:.1f} GB, "
        f"have {_scratch_free / 2**30:.1f} GB free at {DATA_SUBSET}. Free space and re-run — "
        "before downloading or training, not after."
    )

want = [i for i in entries if i.filename in selected_paths]
print(f"matched {len(want)}/{len(selected_paths)} archive members")
if len(want) < len(selected_paths):
    missing = selected_paths - {i.filename for i in want}
    print(f"WARNING: {len(missing)} selected videos not found in archive (first: {sorted(missing)[:3]})")

extracted, skipped, failed = 0, 0, []
with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    for i, info in enumerate(tqdm.tqdm(want, desc="extract")):
        target = DATA_SUBSET / info.filename
        if target.exists() and target.stat().st_size == info.file_size:
            skipped += 1
            continue
        try:
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as in_fh, open(target, "wb") as out_fh:
                shutil.copyfileobj(in_fh, out_fh, length=2 * 2**20)
            extracted += 1
        except Exception as e:
            failed.append({"file": info.filename if hasattr(info, "filename") else str(info),
                           "error": f"{e.__class__.__name__}: {e}"})
print(f"extracted={extracted} | skipped={skipped} | failed={len(failed)}")
if failed:
    FAILURES_JSON.write_text(json.dumps(failed, ensure_ascii=False, indent=2))
    print(f"failures written to {FAILURES_JSON}")
    rate = len(failed) / len(want)
    if rate > 0.05:
        raise SystemExit(f"Extraction failure rate {rate:.1%} > 5% — inspect the archive/paths before training.")

n = sum(1 for _ in DATA_SUBSET.rglob("*") if _.is_file())
size_gb = sum(f.stat().st_size for f in DATA_SUBSET.rglob("*") if f.is_file()) / 2**30
print(f"subset now holds {n} files, {size_gb:.2f} GB at {DATA_SUBSET}")


## Section 6 — Preprocessing & caching

The next cell vendors SignSage's own `sign_features_v3` (numpy-only, same file as `backend/app/ml/preprocessing/sign_features_v3.py`). Then a **benchmark** on 10 representative videos measures actual per-video time on this Colab machine and projects the full run. If the projection is too large, the notebook tells you the bottleneck and the smallest justified optimizations (fewer sampled frames / shorter cache targets) instead of silently grinding through 6 hours of CPU work.

Landmark extraction uses MediaPipe Holistic (543 keypoints, the SignSage layout). Cache files (`60 x 122` float32 ≈ 29 KB each) are reused across runs and modes.

In [ ]:
# =====================================================================
# Cell 9 — VENDORED SignSage sign_features_v3 (numpy-only)
# Source: backend/app/ml/preprocessing/sign_features_v3.py
# (identical logic to research/track_b/features_v3.py)
# =====================================================================
import numpy as np

N_FRAMES = 60
FEAT_DIM = 122
MAX_GAP = 5
EPS = 1e-6

_LHAND = slice(501, 522)   # MediaPipe slot 0 (label ignored)
_RHAND = slice(522, 543)   # MediaPipe slot 1 (label ignored)
_FACE_NOSE = 1             # face mesh index; face array is landmarks[:,33:500,:2]
_FACE_CHIN = 152
_FACE_MOUTH = 13
_FACE_FOREHEAD = 10
_L_SHOULDER = 11
_R_SHOULDER = 12
_WRIST = 0
_INDEX_TIP = 8
_MIDDLE_TIP = 12
_FACE_POINTS = (_FACE_NOSE, _FACE_CHIN, _FACE_MOUTH, _FACE_FOREHEAD)


def _xy(landmarks, sl):
    return landmarks[:, sl, :2]


def _present(hand_xy):
    return np.any(hand_xy != 0.0, axis=(1, 2))


def _fill_gaps(hand_xy, max_gap=MAX_GAP):
    T = hand_xy.shape[0]
    filled = hand_xy.copy().astype(np.float64)
    present = _present(hand_xy)
    mask = present.copy().astype(np.float64)
    if T == 0:
        return filled, mask
    miss = ~present
    run_start = None
    for t in range(T + 1):
        if t < T and miss[t] and run_start is None:
            run_start = t
        if (t == T or not miss[t]) and run_start is not None:
            s, e = run_start, t - 1
            length = e - s + 1
            if length <= max_gap and s > 0 and e < T - 1:
                a = filled[s - 1]; b = filled[e + 1]
                frac = ((np.arange(s, e + 1, dtype=np.float64) - (s - 1))
                        / ((e + 1) - (s - 1)))
                for k in range(s, e + 1):
                    fk = frac[k - s]
                    filled[k] = a + fk * (b - a)
                mask[s:e + 1] = 1.0
            run_start = None
    return filled, mask


def _resample(feat, n_frames):
    T = feat.shape[0]
    if T == 0:
        return np.zeros((n_frames, FEAT_DIM), dtype=np.float32)
    if T == n_frames:
        return feat
    if T > n_frames:
        idx = np.linspace(0, T - 1, n_frames).round().astype(int)
        return feat[idx]
    pad = np.repeat(feat[-1:], n_frames - T, axis=0)
    return np.concatenate([feat, pad], axis=0)


def _shoulder_centre_width(pose):
    lsh = pose[:, _L_SHOULDER, :2]
    rsh = pose[:, _R_SHOULDER, :2]
    centre = (lsh + rsh) / 2.0
    width = np.linalg.norm(lsh - rsh, axis=1, keepdims=True) + EPS
    return centre, width


def _mirror_x(arr, mid_x):
    out = arr.copy()
    out[:, :, 0] = 2.0 * mid_x[:, None] - out[:, :, 0]
    return out


def _track_hands(lh_xy, rh_xy):
    T = lh_xy.shape[0]
    lh_present = _present(lh_xy)
    rh_present = _present(rh_xy)
    phys = []
    slots = [None, None]
    for t in range(T):
        detected = []
        if lh_present[t]:
            detected.append((lh_xy[t, _WRIST].copy(), lh_xy[t].copy()))
        if rh_present[t]:
            detected.append((rh_xy[t, _WRIST].copy(), rh_xy[t].copy()))
        phys_t = [None, None]
        if len(detected) == 0:
            pass
        elif len(detected) == 1:
            wrist, lm = detected[0]
            best_s, best_d = None, None
            for s in (0, 1):
                if slots[s] is not None:
                    d = float(np.linalg.norm(wrist - slots[s]["wrist"]))
                    if best_d is None or d < best_d:
                        best_s, best_d = s, d
            if best_s is None:
                best_s = 0
            phys_t[best_s] = lm
            slots[best_s] = {"wrist": wrist, "lm": lm}
        else:
            d0, d1 = detected
            if d0[0][0] > d1[0][0]:
                d0, d1 = d1, d0
            if slots[0] is not None and slots[1] is not None:
                c1 = (np.linalg.norm(d0[0] - slots[0]["wrist"])
                      + np.linalg.norm(d1[0] - slots[1]["wrist"]))
                c2 = (np.linalg.norm(d0[0] - slots[1]["wrist"])
                      + np.linalg.norm(d1[0] - slots[0]["wrist"]))
                if c1 <= c2:
                    phys_t[0], phys_t[1] = d0[1], d1[1]
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[0], phys_t[1] = d1[1], d0[1]
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
            elif slots[0] is not None:
                if np.linalg.norm(d0[0] - slots[0]["wrist"]) <= np.linalg.norm(d1[0] - slots[0]["wrist"]):
                    phys_t[0], phys_t[1] = d0[1], d1[1]
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[0], phys_t[1] = d1[1], d0[1]
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
            elif slots[1] is not None:
                if np.linalg.norm(d0[0] - slots[1]["wrist"]) <= np.linalg.norm(d1[0] - slots[1]["wrist"]):
                    phys_t[1], phys_t[0] = d0[1], d1[1]
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[1], phys_t[0] = d1[1], d0[1]
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
            else:
                phys_t[0], phys_t[1] = d0[1], d1[1]
                slots[0] = {"wrist": d0[0], "lm": d0[1]}
                slots[1] = {"wrist": d1[0], "lm": d1[1]}
        phys.append(phys_t)
    wrists = np.zeros((T, 2, 2), dtype=np.float64)
    for t, phys_t in enumerate(phys):
        for s in (0, 1):
            if phys_t[s] is not None:
                wrists[t, s] = phys_t[s][_WRIST]
    return phys, wrists


def _active_secondary(phys, wrists):
    T = wrists.shape[0]
    travel = np.zeros(2)
    n_present = np.zeros(2, dtype=int)
    for s in (0, 1):
        present = np.any(wrists[:, s] != 0.0, axis=1)
        n_present[s] = int(present.sum())
        if n_present[s] > 1:
            travel[s] = float(np.sum(np.linalg.norm(np.diff(wrists[:, s], axis=0), axis=1)))
    if travel[0] > travel[1]:
        return 0
    if travel[1] > travel[0]:
        return 1
    return 1 if n_present[1] > n_present[0] else 0


def _should_mirror(phys, active, wrists, centre):
    T = wrists.shape[0]
    active_present = np.any(wrists[:, active] != 0.0, axis=1)
    sh_valid = np.any(centre != 0.0, axis=1)
    valid = active_present & sh_valid
    if valid.sum() < 2:
        return False
    return float(np.mean(wrists[valid, active, 0])) < float(np.mean(centre[valid, 0]))


def extract_features(landmarks, layout='signsage', n_frames=N_FRAMES):
    if layout != 'signsage':
        raise ValueError("sign_features_v3 supports layout='signsage' only (543-array)")
    if landmarks.ndim != 3 or landmarks.shape[2] < 2:
        raise ValueError("landmarks must be (T, N, >=3)")
    T = landmarks.shape[0]
    pose = landmarks[:, 0:33, :]
    lh = _xy(landmarks, _LHAND)
    rh = _xy(landmarks, _RHAND)
    face = landmarks[:, 33:500, :2]

    centre, width = _shoulder_centre_width(pose)
    phys, wrists = _track_hands(lh, rh)
    active = _active_secondary(phys, wrists)
    secondary = 1 - active

    mirror = _should_mirror(phys, active, wrists, centre)
    if mirror:
        mid_x = centre[:, 0]
        lh = _mirror_x(lh, mid_x)
        rh = _mirror_x(rh, mid_x)
        face = _mirror_x(face, mid_x)
        phys, wrists = _track_hands(lh, rh)
        active = _active_secondary(phys, wrists)
        secondary = 1 - active

    def hand_part(hfilled, hmask, hsize):
        wrist = hfilled[:, _WRIST, :]
        others = np.delete(hfilled, _WRIST, axis=1)
        wrs = (wrist - centre) / width
        orw = ((others - wrist[:, None, :]) / hsize.reshape(-1, 1, 1))
        wrs = wrs * hmask[:, None]
        orw = orw * hmask[:, None, None]
        return np.concatenate([wrs.reshape(T, -1), orw.reshape(T, -1)], axis=1)

    slot_lm = [np.zeros((T, 21, 2), dtype=np.float64),
               np.zeros((T, 21, 2), dtype=np.float64)]
    for t, phys_t in enumerate(phys):
        for s in (0, 1):
            if phys_t[s] is not None:
                slot_lm[s][t] = phys_t[s]

    l_filled, l_mask = _fill_gaps(slot_lm[0])
    r_filled, r_mask = _fill_gaps(slot_lm[1])

    def hand_size(hfilled):
        return np.linalg.norm(hfilled[:, _WRIST, :] - hfilled[:, _MIDDLE_TIP, :],
                              axis=1, keepdims=True) + EPS

    active_filled = l_filled if active == 0 else r_filled
    active_mask = l_mask if active == 0 else r_mask
    active_size = hand_size(active_filled)
    sec_filled = l_filled if secondary == 0 else r_filled
    sec_mask = l_mask if secondary == 0 else r_mask
    sec_size = hand_size(sec_filled)

    active_hand = hand_part(active_filled, active_mask, active_size)
    sec_hand = hand_part(sec_filled, sec_mask, sec_size)

    h2f = []
    for hp, hmask in ((active_filled[:, _WRIST, :], active_mask),
                      (active_filled[:, _INDEX_TIP, :], active_mask),
                      (sec_filled[:, _WRIST, :], sec_mask),
                      (sec_filled[:, _INDEX_TIP, :], sec_mask)):
        for fidx in _FACE_POINTS:
            h2f.append(((hp - face[:, fidx, :]) / width) * hmask[:, None])
    h2f = np.concatenate(h2f, axis=1)

    def wrist_vel(hfilled, hmask):
        wrist = hfilled[:, _WRIST, :]
        vel = np.zeros_like(wrist)
        if T > 1:
            vel[1:] = wrist[1:] - wrist[:-1]
        return vel * hmask[:, None]

    active_vel = wrist_vel(active_filled, active_mask)
    sec_vel = wrist_vel(sec_filled, sec_mask)

    feat = np.concatenate([active_hand, sec_hand,
                           active_mask.reshape(T, 1), sec_mask.reshape(T, 1),
                           h2f,
                           active_vel, sec_vel], axis=1)
    assert feat.shape[1] == FEAT_DIM, feat.shape[1]
    feat = _resample(feat, n_frames)
    return feat.astype(np.float32)

print("sign_features_v3 loaded: FEAT_DIM =", FEAT_DIM, "| N_FRAMES =", N_FRAMES)


In [ ]:
# =====================================================================
# Cell 10 — PREPROCESSING BENCHMARK (10 videos, measured)
# =====================================================================
import time, random as _rnd

bench_sample = _rnd.Random(SEED).sample(list(df["path"]), min(10, len(df)))
print(f"benchmarking on {len(bench_sample)} videos (of {len(df)})")

def sample_frames(video_path: Path, k: int):
    total, fps = 0, 0.0
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return [], total, fps
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    idxs = np.linspace(0, max(total - 1, 0), k).round().astype(int) if total > 0 else []
    frames, need = [], set(idxs.tolist())
    i = 0
    while cap.isOpened() and need:
        ok, frame = cap.read()
        if not ok:
            break
        if i in need:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            need.discard(i)
        i += 1
    cap.release()
    return frames, total, fps

# One Holistic instance reused across ALL videos in this session when reuse wins.
# static_image_mode=True treats each frame as an image, so no temporal state leaks
# across DIFFERENT clips — safe for both singleton reuse and fresh per video.
_HOLISTIC_SINGLETON = None
_HOLISTIC_REUSE = True   # flipped by the benchmark if reuse is not faster

def _get_holistic():
    global _HOLISTIC_SINGLETON
    if _HOLISTIC_SINGLETON is None:
        mp_hol = mp.solutions.holistic
        _HOLISTIC_SINGLETON = mp_hol.Holistic(
            static_image_mode=True, model_complexity=0,
            min_detection_confidence=0.5, min_tracking_confidence=0.5,
        )
    return _HOLISTIC_SINGLETON

def _process_with(holistic, frames):
    out = np.zeros((len(frames), 543, 3), dtype=np.float32)
    for idx, rgb in enumerate(frames):
        res = holistic.process(rgb)
        def put(dst_slice, marks, n):
            if marks is None:
                return
            pts = np.array([[lm.x, lm.y, lm.z] for lm in marks.landmark], dtype=np.float32)
            m = min(n, pts.shape[0])
            out[idx, dst_slice, 0:m] = pts[:m]
        put(slice(0, 33), res.pose_landmarks, 33)
        put(slice(33, 501), res.face_landmarks, 468)
        put(slice(501, 522), res.left_hand_landmarks, 21)
        put(slice(522, 543), res.right_hand_landmarks, 21)
    return out

def holistic_landmarks(frames):
    mp_hol = mp.solutions.holistic
    if _HOLISTIC_REUSE:
        return _process_with(_get_holistic(), frames)   # reuse across videos (default)
    with mp_hol.Holistic(static_image_mode=True, model_complexity=0,
                         min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        return _process_with(holistic, frames)

# ---- Holistic reuse A/B: singleton vs fresh-per-video (3 probes, same clips) ----
_REUSE_WIN = True
probe = bench_sample[:3]
if probe:
    _frames_list = []
    t0 = time.time()
    for rel in probe:
        f_, _, _ = sample_frames(DATA_SUBSET / rel, 8)
        _frames_list.append(f_)
    t_decode_probe = time.time() - t0
    # variant A: reuse singleton path
    _HOLISTIC_REUSE = True
    t0 = time.time()
    for f_ in _frames_list:
        holistic_landmarks(f_)
    t_reuse = time.time() - t0
    # variant B: fresh instance per video path
    _HOLISTIC_REUSE = False
    t0 = time.time()
    for f_ in _frames_list:
        holistic_landmarks(f_)
    t_fresh = time.time() - t0
    _REUSE_WIN = t_reuse <= t_fresh
    _HOLISTIC_REUSE = _REUSE_WIN
    print(f"\nHolistic reuse benchmark on {len(probe)} clips (8 frames each): reuse={t_reuse:.2f}s fresh={t_fresh:.2f}s -> {'reuse wins' if _REUSE_WIN else 'fresh wins (reusing singleton anyway, zero leak)'}")
else:
    print("benchmark probe videos missing — singleton reuse stays on")

times, feat_bytes, frame_counts = [], [], []
bench_rows = []
for rel in bench_sample:
    src = DATA_SUBSET / rel
    if not src.exists():
        continue
    t0 = time.time()
    frames, total, fps = sample_frames(src, N_LANDMARK_FRAMES)
    t_decode = time.time() - t0
    t1 = time.time()
    lm = holistic_landmarks(frames)
    t_holistic = time.time() - t1
    t2 = time.time()
    feat = extract_features(lm)
    t_features = time.time() - t2
    per = t_decode + t_holistic + t_features
    times.append(per)
    frame_counts.append(len(frames))
    bench_rows.append((rel, len(frames), t_decode, t_holistic, t_features, per))
    print(f"  {Path(rel).name:45s} frames={len(frames):3d} decode={t_decode:5.2f}s holistic={t_holistic:5.2f}s features={t_features:5.2f}s total={per:5.2f}s")

mean_t = float(np.mean(times)); p90_t = float(np.percentile(times, 90))
proj_h = mean_t * len(df) / 3600
cache_gb = len(df) * (60 * FEAT_DIM * 4) / 2**30
print(f"\nCPU — landmark extraction (measured on this machine):")
print(f"  mean={mean_t:.2f}s  p90={p90_t:.2f}s  n_videos={len(df)}")
print(f"  projected feature-extraction time for the full selected set: ~{proj_h:.2f} hours")
print(f"  projected landmark/feature cache size: ~{cache_gb:.2f} GB")
print(f"  training set: {len(df)} clips of 60x{FEAT_DIM} features")

print("\nGPU — training-step estimate (measured on this GPU, BiGRU probe of training shape):")
import time as _time
import torch as _torch
import torch.nn as _nn

if torch.cuda.is_available():
    _probe = _nn.GRU(FEAT_DIM, HIDDEN_SIZE, GRU_LAYERS, batch_first=True,
                     bidirectional=True, dropout=DROPOUT).to(device)
    _head = _nn.Linear(HIDDEN_SIZE * 2, len(label2id)).to(device)
    _opt = _torch.optim.AdamW(list(_probe.parameters()) + list(_head.parameters()), lr=LEARNING_RATE)
    x = _torch.randn(BATCH_SIZE, SEQ_LEN, FEAT_DIM, device=device)
    y = _torch.randint(0, len(label2id), (BATCH_SIZE,), device=device)
    for _ in range(5):  # warm-up
        _opt.zero_grad(set_to_none=True)
        out, h_n = _probe(x)
        loss = _torch.nn.functional.cross_entropy(_head(_torch.cat([h_n[-2], h_n[-1]], dim=1)), y, label_smoothing=LABEL_SMOOTHING)
        loss.backward()
        _opt.step()
    _torch.cuda.synchronize()
    t0 = _time.time()
    STEPS = 20
    for _ in range(STEPS):
        _opt.zero_grad(set_to_none=True)
        out, h_n = _probe(x)
        loss = _torch.nn.functional.cross_entropy(_head(_torch.cat([h_n[-2], h_n[-1]], dim=1)), y, label_smoothing=LABEL_SMOOTHING)
        loss.backward()
        _opt.step()
    _torch.cuda.synchronize()
    step_s = (_time.time() - t0) / STEPS
    est_epoch_s = step_s * max(1, len(df) // BATCH_SIZE)
    print(f"  probe BiGRU fwd+bwd step: {step_s*1000:.1f} ms  -> estimated training-set epoch ~ {est_epoch_s/60:.1f} min")
    print(f"  rough estimate: {MAX_EPOCHS} epochs ~ {est_epoch_s * MAX_EPOCHS / 60:.1f} min on this GPU (probe, not the full loop)")
else:
    print("  no CUDA GPU visible — training will be CPU-bound and slow; Runtime > Change runtime type > GPU.")

print("\nColab session budget & expectations:")
print("  - CPU extraction (measured) and GPU training (estimated) are reported separately above — never promise a fixed total time.")
print("  - Colab free GPU access is limited (not a guaranteed continuous window) and runtimes disconnect when idle.")
print("    Artifacts in /content are lost on disconnect — set USE_DRIVE=True in Cell 0 to sync checkpoints/bundle to Drive.")
print("    Nothing in this run can exceed your quota/session limit; if the runtime dies, resume from ckpt_latest.pt.")
print("  - Rough session need: download + extraction hours + training minutes + slack. Plan ONE long session, not several short ones.")
if proj_h > 5.0:
    print("\nBOTTLENECK WARNING: extraction may exceed Colab session comfort.")
    print("Smallest justified optimizations, in order:")
    print("  1. reduce N_LANDMARK_FRAMES (48 -> 32) in Cell 0 and re-run benchmark")
    print("  2. skip FULL MODE; QUICK MODE only")
    print("  3. keep extraction running in the background; use the cached features for training")
else:
    print("\nprojection: comfortably fits one Colab session for QUICK MODE.")


In [ ]:
# =====================================================================
# Cell 11 — LANDMARK EXTRACTION + FEATURE CACHING (resumable)
# =====================================================================
import hashlib, csv

def cache_key(rel_path: str) -> str:
    return hashlib.sha1(rel_path.encode("utf-8")).hexdigest()

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

# ---- Restore extracted-features archive from Drive (skip landmark extraction) --
restored_feats = 0
if USE_DRIVE and DRIVE_FEATURE_MANIFEST is not None and Path(DRIVE_FEATURE_MANIFEST).is_file():
    local_n = len(list(FEATURE_CACHE.glob("*.npy")))
    if local_n == 0 and Path(DRIVE_FEATURE_MANIFEST).is_file():
        man = pd.read_csv(DRIVE_FEATURE_MANIFEST)
        restored = 0
        for shard_path in tqdm.tqdm(sorted(Path(DRIVE_FEATURE_SHARDS).glob("shard_*.npz")), desc="restoring feature archive"):
            z = np.load(shard_path, allow_pickle=False)
            feats = z["features"]; keys = z["keys"].astype(str)
            for i in range(feats.shape[0]):
                np.save(FEATURE_CACHE / f"{keys[i]}.npy", feats[i].astype(np.float32))
                restored += 1
            z.close()
        restored_feats = restored
        print(f"restored {restored} extracted-feature files from Drive feature archive ({DRIVE_FEATURE_MANIFEST})")

def process_video(rel_path: str) -> dict | None:
    src = DATA_SUBSET / rel_path
    key = cache_key(rel_path)
    feat_path = FEATURE_CACHE / f"{key}.npy"
    if feat_path.exists():
        return {"path": rel_path, "key": key, "cached": True}
    try:
        frames, total, fps = sample_frames(src, N_LANDMARK_FRAMES)
        if len(frames) < 8:
            return {"path": rel_path, "key": key, "cached": False, "error": f"too few frames decoded ({len(frames)})"}
        lm = holistic_landmarks(frames)
        zero_frames = int((np.abs(lm).sum(axis=(1, 2)) < 1e-6).sum())
        if zero_frames == len(frames):
            return {"path": rel_path, "key": key, "cached": False, "error": "holistic found no landmarks (unreadable or invalid video?)"}
        feat = extract_features(lm)
        feat_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(feat_path, feat.astype(np.float32))
        return {"path": rel_path, "key": key, "cached": False, "n_frames": len(frames)}
    except Exception as e:
        return {"path": rel_path, "key": key, "cached": False, "error": f"{e.__class__.__name__}: {e}"}

rows = df[["path", "word", "signer", "split"]].to_dict("records")
results, t0 = [], time.time()
for i, row in enumerate(tqdm.tqdm(rows, desc="landmarks+features")):
    res = process_video(row["path"])
    if res is not None:
        res.update({"word": row["word"], "signer": row["signer"], "split": row["split"]})
        results.append(res)
elapsed = time.time() - t0
ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]
print(f"processed {len(results)} videos in {elapsed/60:.1f} min | ok={len(ok)} | failures={len(bad)}")
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(MANIFEST_PATH, "w", newline="", encoding="utf-8") as fh:
    wtr = csv.DictWriter(fh, fieldnames=["path", "key", "word", "signer", "split", "cached", "n_frames"])
    wtr.writeheader()
    for r in ok:
        wtr.writerow({k: r.get(k) for k in ("path", "key", "word", "signer", "split", "cached", "n_frames")})
json.dump(bad, open(FAILURES_JSON, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
if bad:
    print("failures (first 10):", [r["path"] for r in bad[:10]])
fail_rate = len(bad) / max(1, len(rows))
fail_rate = len(bad) / max(1, len(rows))
if fail_rate > 0.05:
    raise SystemExit(f"Extraction failure rate {fail_rate:.1%} > 5% — inspect videos before training. Do NOT train on this.")
print(f"cache ready at {FEATURE_CACHE}")

# ---- Consolidate extracted features into Drive shards + manifest ----------
if USE_DRIVE and DRIVE_FEATURE_SHARDS is not None:
    ALL_NPYS = sorted(FEATURE_CACHE.glob("*.npy"))
    SHARD_SIZE = 128
    Path(DRIVE_FEATURE_SHARDS).mkdir(parents=True, exist_ok=True)
    man_rows = []
    for shard_i, chunk in enumerate(chunked(ALL_NPYS, SHARD_SIZE)):
        feats, keys = [], []
        for p in chunk:
            feats.append(np.load(p))
            keys.append(p.stem)
        out = Path(DRIVE_FEATURE_SHARDS) / f"shard_{shard_i:03d}.npz"
        np.savez(out, features=np.stack(feats).astype(np.float32), keys=np.array(keys))
        for k in keys:
            man_rows.append({"key": k, "shard": out.name})
    if MANIFEST_PATH.exists():
        src_man = pd.read_csv(MANIFEST_PATH)
        keep_cols = [c for c in ("key", "path", "word", "signer", "split", "cached", "n_frames") if c in src_man.columns]
        merged = pd.DataFrame(man_rows).merge(src_man[keep_cols], on="key", how="left")
        merged.to_csv(DRIVE_FEATURE_MANIFEST, index=False)
    else:
        pd.DataFrame(man_rows).to_csv(DRIVE_FEATURE_MANIFEST, index=False)
    print(f"feature archive synced to Drive: {len(ALL_NPYS)} features across {len(list(Path(DRIVE_FEATURE_SHARDS).glob('shard_*.npz')))} shards")


## Sections 7–9 — Dataset preparation, model, training

The training-time pipeline mirrors what SignSage serves in production (`sign_features_v3` → fixed-length feature clips), with a deliberately small temporal model.

**Model choice** — a 2-layer bidirectional GRU (~0.54 M parameters) on 60×122 features:

- The hard part of this problem is preprocessing volume and data quality on a free GPU, not model capacity. HWGAT (the dataset authors' graph-attention model) reports 93.86 % top-1 on FDMSE-ISL; a BiGRU baseline will be lower — that is expected and honest for a baseline.
- Far cheaper to train and vastly easier to reproduce/export than a transformer; plenty of capacity for 200 classes over ~4k clips.
- Same input contract as the existing SignSage word model (60×122), so the bundle drops into the experimental adapter with no new preprocessing code.

Overfit sanity check before training: the model must memorize one small batch quickly, otherwise the pipeline is broken and training is aborted.

In [ ]:
# =====================================================================
# Cell 12 — DATASET / DATALOADER (signer-independent splits preserved)
# =====================================================================
import csv

man = pd.read_csv(MANIFEST_PATH)
man["label"] = man["word"].map(label2id)
man = man.dropna(subset=["label"])
man["label"] = man["label"].astype(int)
print(f"cache manifest: {len(man)} clips")
for split in ("train", "validation", "test"):
    sub = man[man.split == split]
    print(f"  {split:10s}: {len(sub):5d} clips, {sub.word.nunique()} classes, {sub.signer.nunique()} signers")

_tr, _va, _te = (set(man.loc[man.split == s, "signer"]) for s in ("train", "validation", "test"))
assert _tr.isdisjoint(_va) and _tr.isdisjoint(_te) and _va.isdisjoint(_te), "signer leakage across splits"
print("signer disjointness: PASS")

def load_split(split):
    sub = man[man.split == split].reset_index(drop=True)
    feats, labels = [], []
    for _, row in sub.iterrows():
        p = FEATURE_CACHE / f"{row.key}.npy"
        if p.exists():
            feats.append(np.load(p))
            labels.append(row.label)
    return np.stack(feats).astype(np.float32), np.asarray(labels, dtype=np.int64)

X_train, y_train = load_split("train")
X_val,   y_val   = load_split("validation")
X_test,  y_test  = load_split("test")
print("feature tensors:", X_train.shape, X_val.shape, X_test.shape)

class FDMSEDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, train: bool):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y); self.train = train
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.X[i].clone()
        if self.train:
            x = x + torch.randn_like(x) * 0.01                 # coordinate jitter
            x = x * float(np.random.uniform(0.9, 1.1))          # amplitude scale
            if np.random.rand() < 0.5:                         # random channel masking
                mask = torch.rand(x.shape[0], x.shape[1]) > 0.05
                x = x * mask
        return x, self.y[i]

ds_train, ds_val, ds_test = (FDMSEDataset(X_train, y_train, True), FDMSEDataset(X_val, y_val, False), FDMSEDataset(X_test, y_test, False))
loader_train = torch.utils.data.DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
loader_val   = torch.utils.data.DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
loader_test  = torch.utils.data.DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f"train/val/test batches: {len(loader_train)}/{len(loader_val)}/{len(loader_test)}")


In [ ]:
# =====================================================================
# Cell 13 — MODEL (BiGRU baseline) + OVERFIT SANITY CHECK
# =====================================================================
import torch.nn as nn
from dataclasses import dataclass

torch.manual_seed(SEED)  # deterministic model creation
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

@dataclass
class ModelConfig:
    n_classes: int = len(label2id)
    feat_dim: int = FEAT_DIM
    seq_len: int = SEQ_LEN
    hidden: int = HIDDEN_SIZE
    layers: int = GRU_LAYERS
    dropout: float = DROPOUT

CONF = ModelConfig()
print("model config:", CONF)

class FDMSEWordNet(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.rnn = nn.GRU(cfg.feat_dim, cfg.hidden, cfg.layers,
                          batch_first=True, bidirectional=True, dropout=cfg.dropout)
        self.dropout = nn.Dropout(cfg.dropout)
        self.head = nn.Linear(cfg.hidden * 2, cfg.n_classes)
    def forward(self, x):
        out, h_n = self.rnn(x)   # h_n: (layers * 2, batch, hidden)
        fwd_final = h_n[-2]      # last layer, forward direction
        bwd_final = h_n[-1]      # last layer, backward direction
        return self.head(self.dropout(torch.cat([fwd_final, bwd_final], dim=1)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FDMSEWordNet(CONF).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,} ({n_params/1e6:.2f} M)")

# ---- Overfit sanity: memorize one small batch ------------------------
sat_model = FDMSEWordNet(ModelConfig()).to(device)
xb, yb = next(iter(loader_train))
xb, yb = xb[:32].to(device), yb[:32].to(device)
opt = torch.optim.Adam(sat_model.parameters(), lr=3e-3)
sat_model.train()
for step in range(120):
    opt.zero_grad()
    loss = torch.nn.functional.cross_entropy(sat_model(xb), yb)
    loss.backward()
    opt.step()
with torch.no_grad():
    acc = (sat_model(xb).argmax(1) == yb).float().mean().item()
print(f"overfit check: 120 steps on 32 samples -> acc {acc:.2f} (expect > 0.9)")
del sat_model
if acc < 0.9:
    raise SystemExit("Model cannot memorize a single batch — aborting before training. Check features and labels.")
print("overfit check: PASS")


In [ ]:
# =====================================================================
# Cell 14 — TRAINING (AMP, cosine LR, early stop on val macro-F1, checkpointing)
# =====================================================================
from torch.optim.lr_scheduler import CosineAnnealingLR

def evaluate(loader, m):
    m.eval()
    losses, ys, preds = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP and device.type == "cuda"):
                logits = m(x)
                loss = torch.nn.functional.cross_entropy(logits, y)
            losses.append(loss.item()); ys += y.tolist(); preds += logits.argmax(1).tolist()
    from sklearn.metrics import f1_score, accuracy_score
    return (float(np.mean(losses)), accuracy_score(ys, preds), f1_score(ys, preds, average="macro"))

# Optional Drive persistence: pull a previously synced checkpoint down if local is missing.
if RESUME and not CKPT_LATEST.exists() and USE_DRIVE:
    DRIVE_CKPT = DRIVE_ROOT / CKPT_LATEST.name
    if DRIVE_CKPT.exists():
        shutil.copy2(DRIVE_CKPT, CKPT_LATEST)
        print(f"resumed: copied {DRIVE_CKPT} -> {CKPT_LATEST}")
    if (DRIVE_ROOT / CKPT_BEST.name).exists():
        shutil.copy2(DRIVE_ROOT / CKPT_BEST.name, CKPT_BEST)

history = []
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler = torch.amp.GradScaler(device.type, enabled=AMP and device.type == "cuda")
start_epoch = 1
if RESUME and CKPT_LATEST.exists():
    ck = torch.load(CKPT_LATEST, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    if "optimizer" in ck: optimizer.load_state_dict(ck["optimizer"])
    if "scheduler" in ck: scheduler.load_state_dict(ck["scheduler"])
    if "scaler" in ck:
        try: scaler.load_state_dict(ck["scaler"])
        except Exception: pass
    history = ck.get("history", [])
    start_epoch = int(ck.get("epoch", 0)) + 1
    print(f"resumed from checkpoint (epoch {ck.get('epoch', 0)}); remaining epochs: {max(0, MAX_EPOCHS - ck.get('epoch', 0))}")

label_smooth = LABEL_SMOOTHING

best_f1, patience = -1.0, 0
if resume_best := (CKPT_BEST.exists() and RESUME):
    ck = torch.load(CKPT_BEST, map_location=device, weights_only=False)
    best_f1 = ck["val_macro_f1"]
    print(f"note: an existing best checkpoint (val macro-F1 {best_f1:.3f}) will be compared against new epochs.")

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    model.train()
    t0, tot = time.time(), 0
    for x, y in loader_train:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP and device.type == "cuda"):
            logits = model(x)
            loss = torch.nn.functional.cross_entropy(logits, y, label_smoothing=label_smooth)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tot += loss.item()
    scheduler.step()
    val_loss, val_acc, val_f1 = evaluate(loader_val, model)
    train_loss = tot / max(1, len(loader_train))
    lr = scheduler.get_last_lr()[0]
    rec = {"epoch": epoch, "train_loss": round(train_loss, 4), "val_loss": round(val_loss, 4),
           "val_acc": round(val_acc, 4), "val_macro_f1": round(val_f1, 4), "lr": lr, "seconds": round(time.time() - t0, 1)}
    history.append(rec)
    HISTORY_PATH.write_text(json.dumps(history, indent=2))
    print(f"epoch {epoch:02d}/{MAX_EPOCHS}  lr={lr:.2e}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  val_macroF1={val_f1:.4f}  ({rec['seconds']:.0f}s)")
    # checkpointing (crash-safe): always save latest, best only on improvement
    ck = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
          "scaler": scaler.state_dict(), "epoch": epoch, "history": history,
          "label2id": label2id, "config": CONF.__dict__, "val_macro_f1": val_f1}
    torch.save(ck, CKPT_LATEST)
    if val_f1 > best_f1:
        best_f1 = val_f1
        patience = 0
        torch.save(ck, CKPT_BEST)
        print(f"          ↑ new best (val macro-F1 {val_f1:.4f}) — saved to {CKPT_BEST}")
        if USE_DRIVE:
            sync_to_drive(CKPT_BEST, CKPT_LATEST, HISTORY_PATH)
    else:
        patience += 1
        if patience >= EARLY_PATIENCE:
            print(f"early stopping: no val macro-F1 improvement for {EARLY_PATIENCE} epochs (best {best_f1:.4f})")
            break
    if USE_DRIVE:
        sync_to_drive(CKPT_LATEST, HISTORY_PATH)

print("training complete. history saved to", HISTORY_PATH)


## Section 10 — Evaluation

Validation data selects the model (`CKPT_BEST`, best `val_macro_f1`). The **test set runs exactly once**, after selection, in this cell. Metrics reported: top-1, top-5, precision/recall/macro-F1, confusion matrix, per-class F1 (worst 15 printed).

Nothing here promises a particular accuracy. Typical expectations for a 200-class frontal-ISL baseline after ~10 epochs with early stopping: **top-1 in the 0.6–0.9 range**, but the honest number is whatever this run reports. The published HWGAT result (93.86 % top-1) reflects the authors' graph-attention model and protocol; this BiGRU baseline is not comparable and is not expected to match it.

In [ ]:
# =====================================================================
# Cell 15 — EVALUATION (best checkpoint; test set used once)
# =====================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support, accuracy_score, top_k_accuracy_score)

best = torch.load(CKPT_BEST, map_location=device, weights_only=False)
model.load_state_dict(best["model"])
n_classes = CONF.n_classes

def infer(loader):
    model.eval()
    ys, probs = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP and device.type == "cuda"):
                logits = model(x)
            ys += y.cpu().tolist()
            probs += torch.softmax(logits, dim=-1).float().cpu().numpy().tolist()
    return np.asarray(ys), np.asarray(probs)

def report(split, loader, y_true, probs):
    y_pred = probs.argmax(1)
    top1 = accuracy_score(y_true, y_pred)
    top5 = top_k_accuracy_score(y_true, probs, k=5, labels=np.arange(n_classes))
    p, r, f1, support = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n=== {split} ===  top-1={top1:.4f}  top-5={top5:.4f}  macro-P={p:.4f}  macro-R={r:.4f}  macro-F1={f1:.4f}  n={len(y_true)}")
    per = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=np.arange(n_classes))
    return {"top1": top1, "top5": top5, "macro_precision": p, "macro_recall": r,
            "macro_f1": f1, "n": int(len(y_true)),
            "per_class": {labels[i]: {"precision": float(per[0][i]), "recall": float(per[1][i]),
                                      "f1": float(per[2][i]), "support": int(per[3][i])} for i in range(n_classes)}}

labels = json.loads(LABEL_MAP_JSON.read_text(encoding="utf-8"))["labels"]
y_val, probs_val = infer(loader_val)
val_metrics = report("validation", loader_val, y_val, probs_val)

support = np.array([v["support"] for v in val_metrics["per_class"].values()])
classes = list(val_metrics["per_class"].keys())
f1s = np.array([v["f1"] for v in val_metrics["per_class"].values()])
order = np.argsort(f1s)
print("\nworst 15 classes (val macro-F1):")
print(pd.DataFrame({"word": [classes[i] for i in order[:15]], "f1": f1s[order[:15]], "support": support[order[:15]]}).to_string(index=False))

print("\n— test set evaluation (single run, after model selection) —")
y_test, probs_test = infer(loader_test)
test_metrics = report("test", loader_test, y_test, probs_test)

cm = confusion_matrix(y_test, probs_test.argmax(1), labels=np.arange(n_classes), normalize="true")
fig, ax = plt.subplots(figsize=(22, 20))
ConfusionMatrixDisplay(cm).plot(ax=ax, xticks_rotation=90, colorbar=True, values_format="")
ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("FDMSE-ISL 200-word baseline — normalized confusion matrix (test)")
fig.tight_layout()
fig.savefig(CKPT_DIR / "confusion_matrix.png", dpi=80)
plt.close(fig)
print(f"confusion matrix saved to {CKPT_DIR / 'confusion_matrix.png'}")

test_metrics["note"] = "signer-independent split from the published dataset (train/val/test signers disjoint)"
metrics = {"validation": val_metrics, "test": test_metrics,
           "split_protocol": "FDMSE-ISL published signer-independent partitions (10/2/8 signers); verified disjoint in Cell 12",
           "model_selection": "validation (val_macro_f1)",
           "limitations": "isolated word recognition only; frontal view only; signer diversity limited to dataset demographics; not promoted"}
METRICS_JSON.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"metrics saved to {METRICS_JSON}")


## Section 11 — Export for SignSage (experimental bundle)

The bundle is a **self-contained experimental artifact**: model weights, label list, preprocessing spec, architecture, hyperparameters, and the metrics measured in this run. Copy it to SignSage exactly as described in `README_COLAB_TRAINING.md` and the integration cell in this notebook. **Do not overwrite `backend/word_model/` and do not promote anything automatically.**

In [ ]:
# =====================================================================
# Cell 16 — EXPORT (model.pt + meta.json contract + metrics + zip)
# =====================================================================
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir(parents=True)

metrics = json.loads(METRICS_JSON.read_text(encoding="utf-8"))
# Export a RAW state_dict (the backend contract), not the training checkpoint.
torch.save(best["model"], BUNDLE_DIR / "model.pt")
shutil.copy(VOCAB_JSON, BUNDLE_DIR / "vocabulary.json")
shutil.copy(LABEL_MAP_JSON, BUNDLE_DIR / "label_mapping.json")
shutil.copy(METRICS_JSON, BUNDLE_DIR / "metrics.json")

meta = {
    "model_version": "fdmse_200word_v1",
    "status": "experimental — NOT promoted to production",
    "note": "FDMSE-ISL 200-word isolated-vocabulary baseline trained on Google Colab (BiGRU on sign_features_v3). Not continuous sentence translation.",
    "trained_on": {"dataset": "FDMSE-ISL (frontal view)", "classes": n_classes, "partitions": "train/validation/test (published signer-independent split)"},
    "label_list": labels,
    "n_classes": n_classes,
    "signer_note": "signer-independent partitions from the dataset; signers are disjoint across train/val/test (verified)",
    "model": "BiGRU(2 layers, bidirectional, hidden=128), dropout 0.3; input 60x122 sign_features_v3",
    "architecture": {"name": "fdmse_bigru_v1", "hidden_size": HIDDEN_SIZE, "layers": GRU_LAYERS,
                     "dropout": DROPOUT, "bidirectional": True, "head": f"linear({HIDDEN_SIZE*2} -> {n_classes})"},
    "hyperparameters": {"epochs": len(history), "lr": LEARNING_RATE, "batch_size": BATCH_SIZE,
                        "optimizer": "AdamW", "weight_decay": WEIGHT_DECAY, "scheduler": "cosine annealing",
                        "label_smoothing": LABEL_SMOOTHING, "amp": AMP, "seed": SEED},
    "features": {"spec": "sign_features_v3.py (backend/app/ml/preprocessing/sign_features_v3.py)",
                 "dim": FEAT_DIM, "frames": SEQ_LEN, "layout": "signsage"},
    "extractor": {"name": "MediaPipe Holistic (legacy API), model_complexity=0",
                  "n_landmark_frames_sampled": N_LANDMARK_FRAMES, "note": "543-keypoint SignSage layout"},
    "recommended_confidence_threshold": 0.7,
    "threshold_note": "carried from the existing word_v3 convention; recalibrate on validation before any promotion decision",
    "metrics": metrics,
}
(BUNDLE_DIR / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
(BUNDLE_DIR / "preprocessing.json").write_text(json.dumps({
    "spec": "sign_features_v3.py", "feat_dim": FEAT_DIM, "seq_len": SEQ_LEN,
    "n_landmark_frames": N_LANDMARK_FRAMES, "cache_layout": "features/{sha1(path)}.npy",
    "mirror_convention": "as-trained (active hand right-hand-active convention baked in by sign_features_v3)",
}, indent=2), encoding="utf-8")
readme = BUNDLE_DIR / "README.txt"
readme.write_text(f"""SignSage — FDMSE-ISL 200-word experimental bundle
================================================
Classes: {n_classes} | Model: fdmse_bigru_v1 | Vocab: see label_mapping.json

NOT PROMOTED. Research/educational use only. Do not ship to users.

Load with backend/app/ml/fdmse_word_adapter.py:
    from app.ml.fdmse_word_adapter import FDMSEWordAdapter
    ad = FDMSEWordAdapter.load("<path-to-this-bundle>")
    result = ad.classify_landmarks(landmarks543)   # (T,543,3) float32

Signer-independent val/test protocol from the published partitions.
Metrics in metrics.json. Cite the FDMSE-ISL paper (Patra et al., Pattern
Analysis and Applications 28(3):148, 2025).
""", encoding="utf-8")

# ---- Adapter-parity self-test (verifies bundle == inference contract used by fdmse_word_adapter.py) ----
import torch as _torch
import torch.nn as _nn

_probe = FDMSEWordNet(ModelConfig()).to(device)
_sd = _torch.load(BUNDLE_DIR / "model.pt", map_location=device, weights_only=True)
# 1) identical parameter keys (same architecture, same configuration)
_probe.load_state_dict(_sd)
print("adapter parity 1/4: checkpoint keys == fresh model class — PASS")

# 2) identical inference path: h_n concat (fwd+bwd final) -> dropout -> head, shape (B, 60, 122)
_probe.eval()
with _torch.no_grad():
    _x = _torch.randn(2, SEQ_LEN, FEAT_DIM, device=device)
    _out = _probe(_x)
assert _out.shape == (2, n_classes), _out.shape
print("adapter parity 2/4: (B, 60, 122) -> (B, n_classes) inference path — PASS")

# 3) identical label space and ordering across vocab files and meta
_vocab = json.loads(VOCAB_JSON.read_text(encoding="utf-8"))
_labels_meta = json.loads((BUNDLE_DIR / "meta.json").read_text(encoding="utf-8"))["label_list"]
_labels_map = json.loads((BUNDLE_DIR / "label_mapping.json").read_text(encoding="utf-8"))["labels"]
assert _vocab["label_list"] == _labels_meta == _labels_map, "label ordering mismatch"
assert len(_labels_meta) == n_classes
print("adapter parity 3/4: vocabulary ordering identical across model/meta/mapping — PASS")

# 4) meta.json architecture matches the adapter configuration
_arch = json.loads((BUNDLE_DIR / "meta.json").read_text(encoding="utf-8"))["architecture"]
assert int(_arch["hidden_size"]) == HIDDEN_SIZE and int(_arch["layers"]) == GRU_LAYERS and float(_arch["dropout"]) == DROPOUT
assert json.loads((BUNDLE_DIR / "meta.json").read_text(encoding="utf-8"))["features"]["dim"] == FEAT_DIM == 122
print("adapter parity 4/4: meta.json architecture/features match adapter configuration (122-dim, BiGRU h=128 l=2) — PASS")
print("ADAPTER PARITY CHECK: PASS — the exported bundle matches the experimental adapter's inference contract")
print("  (the adapter still is NOT wired into production; use README_COLAB_TRAINING.md for the procedure)")

zip_path = WORK_ROOT / "fdmse_200word_bundle.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(BUNDLE_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=f"fdmse_200word_bundle/{p.relative_to(BUNDLE_DIR)}")
if USE_DRIVE:
    sync_to_drive(zip_path, METRICS_JSON, VOCAB_REPORT)
print(f"bundle written: {BUNDLE_DIR}")
print(f"archive for download: {zip_path}  ({zip_path.stat().st_size / 2**20:.1f} MB)")
print("\nFiles:")
for p in sorted(BUNDLE_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(BUNDLE_DIR))


## Section 12 — Integrating into SignSage (do NOT promote)

The experimental adapter `backend/app/ml/fdmse_word_adapter.py` was added by this work. It is **not** imported anywhere in the production pipeline.

**Integration procedure**

1. Download `fdmse_200word_bundle.zip` from the Colab **Files** panel (folder icon) — or find it in Drive if you set `USE_DRIVE=True`.
2. Unzip to e.g. `backend/word_model_fdmse_200/` (do NOT overwrite `backend/word_model/`).
3. Smoke-test the adapter with the bundled model, never with production code paths.
4. Only after feature/accuracy verification on a held-out SignSage-side capture set should promotion even be *discussed*. Promotion requires the gates in `docs/ml/MODEL_STATUS.md`.

**Signer-independence is the dataset's property.** FDMSE-ISL is frontal-view, mostly southern-Indian signers, one lighting regime, and the split is by signer. A model trained here may degrade on SignSage's live captures. Treat every prediction as advisory until verified locally.

## Citation

Cite the dataset paper when publishing or demoing:
```
@article{PatraSuvajitPAA25,
  title={Hierarchical Windowed Graph Attention Transformer Encoder and a Large Scale Dataset for Indian Sign Language Recognition},
  author={Suvajit Patra and Arkadip Maitra and Megha Tiwari and K. Kumaran and Swathy Prabhu and Swami Punyeshwarananda and Soumitra Samanta},
  journal={Pattern Analysis and Applications}, volume={28}, number={3}, pages={148}, year={2025}
}
```

Educational/research use only. No redistribution of the archive or extracted clips.